# C16_E01 — Soft sensor con regresión y PLS

**Capítulo:** 16 — Machine Learning, Edge AI y supervisión inteligente de lazos  
**Identificador:** `C16_E01`  
**Objetivo pedagógico:** Construir un soft sensor para una variable difícil de medir.  
**Librerías:** scikit-learn, numpy, matplotlib

## Ejemplo industrial

Estimación de composición de una columna de destilación a partir de temperaturas de bandeja.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Soft sensor con regresión lineal y PLS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
import os
np.random.seed(0)

# Dataset sintético: tres temperaturas de bandeja → composición
N = 200
T1 = np.random.normal(80, 2.0, N)
T2 = np.random.normal(95, 1.5, N)
T3 = np.random.normal(110, 2.5, N)
y = 0.6*T1 - 0.4*T2 + 0.2*T3 + 5.0 + np.random.normal(0, 0.3, N)
X = np.column_stack([T1, T2, T3])

## 2. Ajuste y comparación

In [2]:
lin = LinearRegression().fit(X, y)
pls = PLSRegression(n_components=2).fit(X, y)
y_lin = lin.predict(X)
y_pls = pls.predict(X).ravel()
print(f"R² lineal: {r2_score(y, y_lin):.4f}")
print(f"R² PLS  : {r2_score(y, y_pls):.4f}")

R² lineal: 0.9553
R² PLS  : 0.9553


## 3. Visualización

In [3]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y, y_lin, alpha=0.6, label="Regresión lineal")
ax.scatter(y, y_pls, alpha=0.6, marker='x', label="PLS (2 comp)")
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', alpha=0.5)
ax.set_xlabel("Composición real"); ax.set_ylabel("Soft sensor")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C16_E01 — Soft sensor: dispersión predicción vs real")
out = '../../figures/cap16/fig_C16_F01_soft_sensor.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 4. Interpretación

Un soft sensor estima una variable difícil de medir (composición) a partir de variables fácilmente disponibles (temperaturas de bandeja). La regresión lineal y PLS producen ajustes similares aquí porque las variables son aproximadamente independientes. PLS aporta robustez ante colinealidad cuando el número de variables crece. **Limitación crítica:** un soft sensor nunca sustituye al sensor físico; lo extiende a frecuencias o condiciones donde no llega. La calibración periódica contra el sensor real es obligatoria.